In [3]:
import json
import random
from pathlib import Path

In [ ]:
# Make folderwise metadata files
# Use separate for the lobster


# Root directory containing the four subfolders
ROOT = Path("./training_images")

# Map folder names (or key words in them) to species labels
SPECIES_BY_FOLDER = {
    "Blue_Shark": "blue shark",
    "Lions_Mane_Jellyfish": "lion's mane jellyfish",
    "Mola_Mola": "mola mola",
}

adjectives = [
    "majestic", "striking", "vibrant", "detailed", "cinematic", 
]

actions = [
    "swimming", "gliding", "floating", "drifting", "cruising", "soaring through the water",
]

adverbs = [
    "gracefully", "smoothly", "effortlessly", "fluently", "calmly", "elegantly",
]

trailers = [
    "in natural water", "in the open ocean", "in blue water", "in calm waters",
]


def infer_species_from_folder(folder_name: str) -> str:
    for key, species in SPECIES_BY_FOLDER.items():
        if key.lower() in folder_name.lower():
            return species
    return "marine animal"


def build_prompt(species: str, rng: random.Random, uniqueness_seed: str = "") -> str:
    """Build a single prompt string.

    uniqueness_seed can be e.g. the filename to help diversify choices.
    """
    # Use a fresh seeded RNG per sample for more variety but deterministic if needed
    local_rng = random.Random(rng.random())

    adj = local_rng.choice(adjectives)
    act = local_rng.choice(actions)
    adv = local_rng.choice(adverbs)
    trail = local_rng.choice(trailers)

    # Optional small variations in wording to keep things unique
    base_patterns = [
        "A {adj} {species}, photorealistic, {act} {adv} {trail}, artistic awareness photography style.",
        "Photorealistic image of a {adj} {species} {act} {adv} {trail}, cinematic style.",
        "A highly detailed, photorealistic {species}, {act} {adv} {trail}, documentary style.",
    ]

    pattern = local_rng.choice(base_patterns)
    prompt = pattern.format(
        adj=adj,
        species=species,
        act=act,
        adv=adv,
        trail=trail,
    )

    # Add a light unique descriptor based on filename hash to avoid exact duplicates
    if uniqueness_seed:
        extra_descriptors = [
            "soft lighting", "fine texture detail", "natural color grading",
        ]
        # Use hash of seed to pick an extra descriptor deterministically
        idx = abs(hash(uniqueness_seed)) % len(extra_descriptors)
        prompt += f" Featuring {extra_descriptors[idx]}."

    return prompt


def generate_metadata_for_subfolder(subfolder: Path, rng: random.Random) -> None:
    species = infer_species_from_folder(subfolder.name)
    images = sorted(
        [p for p in subfolder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    )

    if not images:
        print(f"No images found in {subfolder}")
        return

    out_path = subfolder / "metadata.jsonl"
    print(f"Writing {out_path} with {len(images)} records (species: {species})")

    with out_path.open("w", encoding="utf-8") as f:
        for img in images:
            prompt = build_prompt(species, rng, uniqueness_seed=img.name)
            record = {
                "file_name": img.name,
                "text": prompt,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def main():
    rng = random.Random(42)  # change or remove seed if you want non-reproducible randomness

    # Consider only immediate subdirectories (Sharks, Jellyfish, Lobster, Mola Mola)
    for sub in ROOT.iterdir():
        if sub.is_dir():
            generate_metadata_for_subfolder(sub, rng)


if __name__ == "__main__":
    main()

Writing Train_Data/Lions_Mane_Jellyfish/metadata.jsonl with 686 records (species: lion's mane jellyfish)
Writing Train_Data/Mola_Mola/metadata.jsonl with 735 records (species: mola mola)
Writing Train_Data/Americal_Lobster/metadata.jsonl with 110 records (species: marine animal)
Writing Train_Data/Blue_Shark/metadata.jsonl with 1760 records (species: blue shark)


In [ ]:
## Combine smaller metadata files into one
## In the Train_Data folder

import json
from pathlib import Path

root = Path("./Train_Data")
species_dirs = [
    "American_Lobster",
    "Blue_Shark",
    "Lions_Mane_Jellyfish",
    "Mola_Mola",
]

out_path = root / "metadata.jsonl"

with out_path.open("w") as f_out:
    for sp in species_dirs:
        meta_path = root / sp / "metadata.jsonl"
        with meta_path.open() as f_in:
            for line in f_in:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                # Make file_name relative to Train_Data, including subfolder
                rec["file_name"] = f"{sp}/{rec['file_name']}"
                f_out.write(json.dumps(rec) + "\n")

print("Wrote combined metadata to", out_path)


Wrote combined metadata to Train_Data/metadata.jsonl
